# MEN 4338 · Week 13 — Sequential Gaussian simulation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/men4338/notebooks/MEN4338_W13_sequential_gaussian_simulation.ipynb)

**IUGB · BSc Mining Engineering · Geostatistics (lecture & lab)** · Instructor: Laurent Kouadio

Kriging gives the best estimate at each point, but a kriged map is too smooth to look like the real deposit:
its histogram is too narrow, and summing kriged values does not tell how uncertain the total tonnage is.
**Simulation** draws many equally probable versions of the deposit, each honouring the data, the histogram
and the variogram. Their spread **is** the uncertainty. This is how banks and boards are told that a
resource is "18 Mt, between 15 and 21 Mt with 80 % confidence".

**What you will practise:** the normal-score transform · simple kriging in Gaussian space · the sequential
algorithm (random path, conditioning on simulated nodes) · back-transform · probability maps · P90/P50/P10.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/men4338/'
FILES = [('data/bassam_samples.csv', 'bassam_samples.csv'), ('tests/w13/q1.py', 'tests/q1.py'), ('tests/w13/q2.py', 'tests/q2.py'), ('tests/w13/q3.py', 'tests/q3.py'), ('tests/w13/q4.py', 'tests/q4.py'), ('tests/w13/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

In [ ]:
# Toolbox from earlier weeks: run this cell.
import numpy as np

def spherical(h, c0, c, a):
    """Spherical model: nugget c0, partial sill c, range a. gamma(0) = 0."""
    h = np.asarray(h, float)
    r = np.minimum(h / a, 1.0)
    return np.where(h > 0, c0 + c * (1.5 * r - 0.5 * r ** 3), 0.0)

def aniso_h(dx, dy, azimuth, a_major, a_minor):
    """Distance rescaled to the major range: an anisotropic model becomes model(aniso_h(...), c0, c, a_major)."""
    t = np.radians(azimuth)
    u = dx * np.sin(t) + dy * np.cos(t)
    w = dx * np.cos(t) - dy * np.sin(t)
    return np.sqrt(u ** 2 + (w * a_major / a_minor) ** 2)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

samples = pd.read_csv("bassam_samples.csv")
x, y, au = samples["x"].to_numpy(), samples["y"].to_numpy(), samples["au_gpt"].to_numpy()

def gamma_ns(dx, dy):
    """Variogram of the normal scores (sill 1): nugget 0.15 + spherical 0.85, 300 m along N045, 150 m along N135."""
    return spherical(aniso_h(dx, dy, 45, 300, 150), 0.15, 0.85, 300)

## 1. Why a Gaussian transform?

The algorithm relies on a property of the multi-Gaussian model: the conditional distribution at a node given
its neighbours is **normal**, with mean and variance given by **simple kriging**. Grades are not normal, so we
first map them onto a standard normal distribution by **ranks** (normal scores), simulate, and map back.

$$y_i = G^{-1}\!\left(\frac{r_i + 0.5}{n}\right)$$

where $r_i$ is the rank of $z_i$ (0 for the smallest) and $G^{-1}$ is `norm.ppf`.

## 2. The sequential algorithm

1. Transform the data to normal scores.
2. Visit the grid nodes in a **random order**.
3. At each node: simple-krige (mean 0, sill 1) from the nearest data **and previously simulated nodes**;
   draw a value from $N(y^*_{SK}, \sigma^2_{SK})$; add it to the conditioning data.
4. Back-transform all nodes to grades.

A different random seed gives a different, equally probable, realisation.

---
## Exercises

### Q1. Normal scores

Write `nscore(v)` returning the normal scores of an array (ranks with `np.argsort(np.argsort(v))`). Store the
normal scores of `au` in `y_ns` and plot their histogram.

In [ ]:
def nscore(v):
    ...

y_ns = ...
y_ns.mean(), y_ns.var()

In [ ]:
grader.check("q1")

### Q2. Back-transform

Write `back_transform(yv, v)` mapping normal scores `yv` back to grades with `np.interp`, using the pairs
(sorted normal scores of `v`, sorted `v`).

In [ ]:
def back_transform(yv, v):
    ...

np.allclose(back_transform(y_ns, au), au)

In [ ]:
grader.check("q2")

### Q3. Simple kriging in Gaussian space

Write `sk_normal(x0, y0, xs, ys, vs, gamma)` returning `(mean, variance)` of simple kriging with mean 0 and
sill 1 from the points `(xs, ys)` with normal-score values `vs`.

In [ ]:
def sk_normal(x0, y0, xs, ys, vs, gamma):
    ...

sk_normal(30., 20., np.array([0., 60, 0]), np.array([0., 0, 50]), np.array([0.5, -0.2, 1.1]), gamma_ns)

In [ ]:
grader.check("q3")

## 3. The simulation engine

The function below is complete: it uses your `sk_normal`. Read it line by line, then run it.

In [ ]:
def sgs(gx, gy, xd, yd, yd_ns, gamma, seed, nmax=12):
    """One realisation in normal scores on the nodes (gx, gy) (1D arrays), conditioned on (xd, yd, yd_ns)."""
    rng = np.random.default_rng(seed)
    px, py, pv = list(xd), list(yd), list(yd_ns)          # conditioning points: data + simulated nodes
    out = np.zeros(len(gx))
    for node in rng.permutation(len(gx)):                 # random path
        ax, ay, av = np.array(px), np.array(py), np.array(pv)
        k = np.argsort(np.hypot(ax - gx[node], ay - gy[node]))[:nmax]
        m, v = sk_normal(gx[node], gy[node], ax[k], ay[k], av[k], gamma)
        out[node] = rng.normal(m, np.sqrt(max(v, 0.0)))
        px.append(gx[node]); py.append(gy[node]); pv.append(out[node])
    return out

GX, GY = np.meshgrid(np.arange(15, 1200, 30), np.arange(15, 1000, 30))   # 33 rows x 40 columns
print(GX.shape)

### Q4. Twenty realisations

Run `sgs` with seeds 0 to 19 on the nodes `GX.ravel(), GY.ravel()`, back-transform each realisation to grades
and reshape it to `GX.shape`. Store them in `reals` (shape 20 × 33 × 40). This takes from a few seconds to a minute. Draw two
realisations side by side: same data, different deposits.

In [ ]:
reals = ...
reals.shape

In [ ]:
grader.check("q4")

### Q5. Uncertainty on the resource

Each 30 m node stands for 24 300 t. Store in `tonnage_mt` the tonnage above 1.0 g/t of each realisation (Mt),
the P90, P50 and P10 in `t_p90, t_p50, t_p10` (P90 = 10th percentile), and the probability map
`prob_ore` = share of realisations above 1.0 g/t at each node. Draw the histogram of `tonnage_mt` and the map.

In [ ]:
tonnage_mt = ...
t_p90, t_p50, t_p10 = ...
prob_ore = ...
t_p90, t_p50, t_p10

In [ ]:
grader.check("q5")

### Going further (not checked)

* Average the 20 realisations node by node (the "E-type" map) and compare it with a kriging map: which one is smoother?
* Compare `prob_ore` with the indicator kriging map of week 12.
* How much do P90 and P10 move with 50 realisations instead of 20?

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()